# Chapter 9: Exploitation and Post-Exploitation

> "Exploitation is not an end; it is the beginning of understanding what the attacker could do next."

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Explain what exploitation means in a penetration-testing context.
2. Describe common vulnerability classes and their exploitation mechanisms.
3. Explain buffer overflows conceptually and how ASLR, DEP, and stack canaries mitigate them.
4. Describe how Metasploit is structured and how to use it responsibly.
5. Explain privilege escalation techniques for Linux and Windows environments.
6. Describe lateral movement techniques including pass-the-hash and pass-the-ticket.
7. Explain persistence mechanisms and their detection signatures.
8. Document post-exploitation activities with a timeline for the pentest report.

## Key Terms

- **Exploit**: code or technique that triggers a vulnerability to achieve a desired effect.
- **Payload**: the code executed after a successful exploit; commonly a reverse shell or Meterpreter.
- **Shellcode**: machine code injected via a memory-corruption vulnerability.
- **Buffer overflow**: writing beyond an allocated buffer to overwrite adjacent memory.
- **ASLR**: Address Space Layout Randomisation; randomises memory addresses to defeat hardcoded jumps.
- **DEP/NX**: Data Execution Prevention / No-Execute; marks memory pages non-executable.
- **Stack canary**: a random value placed before the return address; checked before function return.
- **ROP**: Return-Oriented Programming; chains existing code gadgets to bypass DEP.
- **Privilege escalation (privesc)**: gaining higher-privilege access than initially obtained.
- **Lateral movement**: using initial access to compromise additional hosts on the network.
- **Pass-the-hash (PtH)**: authenticating with a captured NTLM hash without cracking it.
- **Pass-the-ticket (PtT)**: using a captured Kerberos ticket for authentication.
- **Persistence**: mechanisms maintaining access after session termination or reboot.
- **Meterpreter**: Metasploit's in-memory payload providing an interactive post-exploitation shell.

---

## What Exploitation Is (and Is Not)

### In a Penetration Test

Exploitation in a pentest is the controlled demonstration that a vulnerability can be leveraged to
achieve an attacker's objective. The goal is evidence: a screenshot proving access was obtained, a
hash or flag proving data was reached. Exploitation is not about causing damage; it stops at the
minimum necessary to produce that evidence.

### Ethical Boundaries

A tester who gains a shell on a server does not run commands that could corrupt data, deny service
to users, or access data outside the authorised scope. The rule: minimum necessary access to prove
the point. Evidence of access to the `/etc/passwd` file proves Unix system compromise; it is not
necessary to also access the backup database unless that is specifically authorised.

---

## Common Vulnerability Classes

### Injection Vulnerabilities

Injection vulnerabilities arise when user-controlled data is interpreted as code or a command.
SQL injection passes malicious SQL through an application input to query or modify a database.
Command injection passes shell commands through an application that calls a system function. LDAP
injection, XML injection, and template injection follow the same pattern. All injection attacks
share a root cause: the application fails to separate data from instructions.

#### SQL Injection

A login form that constructs `SELECT * FROM users WHERE username='$u' AND password='$p'` can be
bypassed by entering `' OR 1=1 --` as the username, producing a query that always returns true.
Union-based SQLi extracts data from other tables; blind SQLi infers data through true/false
responses or timing differences (time-based blind).

### Memory Corruption

Memory-corruption vulnerabilities occur when a programme writes to memory it does not own. The
classic example is a stack-based buffer overflow: a fixed-size buffer is filled with attacker-
controlled data that overwrites the saved return address, redirecting execution to attacker-supplied
shellcode.

#### Modern Mitigations

Modern compilers and operating systems layer multiple mitigations:
- **Stack canary**: a random 8-byte value placed between local variables and the saved return
  address. A canary check before function return detects stack corruption.
- **ASLR**: randomises the base addresses of the stack, heap, and libraries at each execution,
  defeating attacks that hardcode addresses.
- **DEP/NX**: marks the stack and heap as non-executable so injected shellcode cannot run.
- **PIE**: Position-Independent Executable; randomises the executable's own base address.

Return-Oriented Programming (ROP) was developed to bypass DEP: instead of injecting shellcode,
the attacker chains together short instruction sequences (gadgets) already present in the binary,
ending each gadget with a `ret` instruction. ROP does not require executable stack or heap.

### Authentication and Session Vulnerabilities

Broken authentication includes: default credentials (admin/admin), weak password policies,
credential stuffing (re-using leaked username/password pairs), missing lockout after failed
attempts, and insecure session tokens (predictable, short, or transmitted in the clear).

---

## Metasploit Framework

### Structure

Metasploit organises attack capabilities into modules:
- **Exploit modules**: trigger a specific vulnerability.
- **Payload modules**: execute after a successful exploit (reverse shell, Meterpreter, cmd).
- **Auxiliary modules**: scanning, enumeration, brute-force, without exploitation.
- **Post modules**: post-exploitation activities (gather credentials, escalate privileges).
- **Encoder modules**: obfuscate payloads to evade signature detection.

### Responsible Use

Metasploit is a penetration-testing tool. Running it against systems without authorisation is a
criminal offence. In authorised engagements, the tester selects the narrowest exploit targeting
the confirmed vulnerable version, uses the safest payload (avoid shellcode that crashes services),
and documents every module and option used.

---

## Privilege Escalation

### Linux Privilege Escalation

Starting from a low-privilege shell, the tester seeks to reach root. Common vectors:

#### SUID/SGID Binaries

Files with the SUID bit execute with the owner's privileges (often root) regardless of who runs
them. `find / -perm -4000 2>/dev/null` finds all SUID files. GTFOBins documents SUID binaries
(vim, find, python) that can be leveraged to spawn a root shell.

#### Sudo Misconfiguration

`sudo -l` lists commands a user can run as root. A misconfigured sudoers entry allowing `sudo vim`
or `sudo python` can trivially escalate to root via the editor's shell escape or Python's
`os.system()`.

#### Kernel Exploits

An unpatched kernel may be vulnerable to a privilege escalation exploit (DirtyCow, PwnKit). These
are high-risk: kernel exploits can crash the system if they fail. Testers document the kernel
version and CVE but often do not run kernel exploits in production environments without explicit
authorisation.

### Windows Privilege Escalation

#### Token Impersonation

Windows uses access tokens to identify the security context of processes. A low-privilege user who
obtains a high-privilege token (via a SeImpersonatePrivilege or SeAssignPrimaryTokenPrivilege
vulnerability) can escalate. Potato exploits (RottenPotato, JuicyPotato) exploit this on service
accounts.

#### Unquoted Service Paths

Windows service executables with unquoted paths containing spaces allow path-traversal privilege
escalation: if `C:\Program Files\Vendor\service.exe` is configured without quotes, Windows will
try `C:\Program.exe` first. Placing a malicious executable at that path causes it to execute as
SYSTEM when the service starts.

---

## Lateral Movement

### Pass-the-Hash

Windows NTLM authentication accepts a hash in place of a password. An attacker who extracts NTLM
hashes from the SAM database, LSASS memory, or network captures using Mimikatz can authenticate
to other machines on the network as that user without cracking the hash. PtH is a fundamental
reason organisations should mandate Credential Guard and disable NTLM where possible.

### Pass-the-Ticket

Kerberos authentication uses tickets. A forged or captured TGT or service ticket can be injected
into a session using Mimikatz's `kerberos::ptt`. A Golden Ticket is a forged TGT signed with the
KRBTGT account's hash; it grants access to any service in the domain and remains valid even after
a user's password change (until the KRBTGT hash is rotated twice).

---

## Persistence

### Common Persistence Mechanisms and Their Detection Signatures

| Mechanism | OS | Detection |
|---|---|---|
| Registry Run keys | Windows | Monitor HKCU/HKLM Run key writes |
| Scheduled tasks | Windows | Event ID 4698 (task created) |
| Cron jobs | Linux | Monitor /etc/cron* and user crontabs |
| Systemd service | Linux | New .service files in /etc/systemd |
| SSH authorised_keys | Linux | Monitor ~/.ssh/authorized_keys writes |
| WMI subscriptions | Windows | WMI activity logs; EDR alerts |
| DLL hijacking | Windows | Monitor DLL loads from user-writable paths |

---

## Why This Matters

Understanding exploitation and post-exploitation from the attacker's perspective is essential for
defenders. Knowing that an unquoted service path allows privilege escalation drives the policy to
scan for and remediate this configuration. Knowing that pass-the-hash works drives the decision to
enable Credential Guard and disable NTLM. Defenders who understand how attacks work implement
controls that address root causes rather than surface symptoms.

---

## News in Focus

Documented ransomware campaigns consistently follow the same post-exploitation playbook: gain initial
access (via phishing or exploitation of public-facing vulnerabilities), escalate privileges within
hours, disable security tools, exfiltrate data, and then deploy the ransomware payload laterally
across the network. The technical capabilities used (pass-the-hash, Golden Tickets, scheduled task
persistence) are thoroughly documented in MITRE ATT&CK and are detectable with properly configured
endpoint detection. The gap between the techniques being known and organisations detecting them
reflects insufficient defensive implementation.

---


In [1]:
# Chapter 9 -- Safe simulation: bounds check, privilege check, MITRE ATT&CK mapper

# ── Safe buffer-bounds simulation (no actual shellcode) ───────────────────────
class SafeBuffer:
    def __init__(self, size):
        self.size = size
        self._data = bytearray(size)
        self.canary = 0xDEADBEEF

    def write(self, data: bytes, offset: int = 0) -> str:
        end = offset + len(data)
        if end > self.size:
            return (f"OVERFLOW DETECTED: attempted to write {len(data)} bytes "
                    f"at offset {offset} into a {self.size}-byte buffer "
                    f"(overflow by {end - self.size} bytes). "
                    f"Canary value would be overwritten.")
        self._data[offset:end] = data
        return f"Write OK: {len(data)} bytes at offset {offset}"

buf = SafeBuffer(64)
print("=== Buffer Bounds Checks ===")
for size, offset in [(20,0),(50,0),(10,55),(64,0),(65,0)]:
    payload = b"A" * size
    result = buf.write(payload, offset)
    print(f"  write({size} bytes @ offset {offset}): {result}")

# ── Privilege escalation checker ──────────────────────────────────────────────
print("\n=== Linux Privilege Escalation Checks (simulation) ===")
checks = [
    ("Sudo -l reveals vim or python",     True,  "GTFOBins shell escape: sudo vim -> :!/bin/bash"),
    ("SUID bit on /usr/bin/find",         True,  "find . -exec /bin/bash -p \\; escalates to owner UID"),
    ("Writable /etc/passwd",              False, "Add root-equivalent entry"),
    ("Kernel 5.8 (CVE-2021-4034 PwnKit)", True,  "Local root via polkit pkexec; patch immediately"),
    ("World-writable cron script",        True,  "Modify cron script to spawn reverse shell as root"),
]
escalation_paths = []
for check, vulnerable, technique in checks:
    status = "VULNERABLE" if vulnerable else "OK       "
    print(f"  [{status}] {check}")
    if vulnerable:
        escalation_paths.append(f"  -> {technique}")

print("\n  Escalation paths found:")
for p in escalation_paths:
    print(p)

# ── MITRE ATT&CK technique mapper ─────────────────────────────────────────────
print("\n=== MITRE ATT&CK Technique Mapping ===")
attack_map = {
    "T1059.001": ("Command and Scripting Interpreter: PowerShell", "Execution"),
    "T1078":     ("Valid Accounts",                               "Persistence, Defence Evasion"),
    "T1550.002": ("Use Alternate Auth Material: Pass-the-Hash",   "Lateral Movement"),
    "T1558.001": ("Steal or Forge Kerberos Tickets: Golden Ticket","Credential Access"),
    "T1053.005": ("Scheduled Task/Job: Scheduled Task",           "Persistence, Privilege Escalation"),
    "T1055":     ("Process Injection",                            "Defence Evasion, Privilege Escalation"),
}
print(f"  {'TTP ID':<12} {'Name':<48} {'Tactic'}")
print("  " + "-"*80)
for tid, (name, tactic) in attack_map.items():
    print(f"  {tid:<12} {name:<48} {tactic}")


=== Buffer Bounds Checks ===
  write(20 bytes @ offset 0): Write OK: 20 bytes at offset 0
  write(50 bytes @ offset 0): Write OK: 50 bytes at offset 0
  write(10 bytes @ offset 55): OVERFLOW DETECTED: attempted to write 10 bytes at offset 55 into a 64-byte buffer (overflow by 1 bytes). Canary value would be overwritten.
  write(64 bytes @ offset 0): Write OK: 64 bytes at offset 0
  write(65 bytes @ offset 0): OVERFLOW DETECTED: attempted to write 65 bytes at offset 0 into a 64-byte buffer (overflow by 1 bytes). Canary value would be overwritten.

=== Linux Privilege Escalation Checks (simulation) ===
  [VULNERABLE] Sudo -l reveals vim or python
  [VULNERABLE] SUID bit on /usr/bin/find
  [OK       ] Writable /etc/passwd
  [VULNERABLE] Kernel 5.8 (CVE-2021-4034 PwnKit)
  [VULNERABLE] World-writable cron script

  Escalation paths found:
  -> GTFOBins shell escape: sudo vim -> :!/bin/bash
  -> find . -exec /bin/bash -p \; escalates to owner UID
  -> Local root via polkit pkexec; patch imm

## Review Questions (MCQ)

**Q1.** A stack canary mitigates buffer overflows by:
A. Preventing writing to the stack  B. Detecting stack corruption before function return  C. Randomising memory addresses  D. Marking the stack non-executable

**Q2.** Return-Oriented Programming (ROP) bypasses which mitigation?
A. Stack canaries  B. ASLR  C. DEP/NX  D. PIE

**Q3.** In Metasploit, the module that executes after a successful exploit is called:
A. Encoder  B. Auxiliary  C. Payload  D. Post

**Q4.** Pass-the-Hash works because:
A. NTLM accepts a hash in place of a password for authentication  B. Windows stores passwords in plaintext  C. Kerberos uses MD5 for ticket signing  D. LSASS can be read without privileges

**Q5.** A SUID binary is dangerous in privilege escalation because:
A. It is always world-writable  B. It executes with the file owner's (often root) privileges  C. It is visible only to root  D. It bypasses the firewall

**Q6.** Which command lists files that a user can run with sudo?
A. `whoami`  B. `id`  C. `sudo -l`  D. `find / -perm -4000`

**Q7.** A Golden Ticket attack requires which account's hash?
A. Domain Administrator  B. KRBTGT  C. Local Administrator  D. Guest

**Q8.** Unquoted service paths allow privilege escalation on Windows because:
A. The service runs as a guest  B. Windows will execute a malicious file placed earlier in the path  C. The registry key is world-writable  D. The service runs without a security token

**Q9.** Which Windows Event ID indicates a new scheduled task was created?
A. 4624  B. 4776  C. 4698  D. 4688

**Q10.** The MITRE ATT&CK technique T1550.002 describes:
A. Spear phishing  B. Pass-the-Hash  C. Golden Ticket  D. PowerShell execution

*Answers: Q1 B, Q2 C, Q3 C, Q4 A, Q5 B, Q6 C, Q7 B, Q8 B, Q9 C, Q10 B.*

## Lab Assignment

**Part A -- Buffer overflow simulation**: Using the `SafeBuffer` class above, add a canary-check method that returns True if the canary value is intact and False if overwritten. Simulate three writes: a safe write, a write that overflows by exactly 1 byte, and a write that overwrites the simulated return address. Print the canary status after each write.

**Part B -- Linux privesc audit**: On a Linux VM you own, run `find / -perm -4000 2>/dev/null` and list all SUID binaries. Look up three of them on GTFOBins. For each, describe whether a privilege escalation path exists and what it is.

**Part C -- ATT&CK mapping**: For a fictional incident where an attacker used phishing for initial access, then ran Mimikatz to extract hashes, authenticated via PtH to three workstations, and created a scheduled task for persistence, map each action to the MITRE ATT&CK technique ID and tactic. Present the full kill chain.

**Part D -- Persistence detection**: List the five persistence mechanisms from the chapter table. For each, write the specific command or detection rule (Sigma, Sysmon, or EventID) that would alert a SOC analyst to its creation.

## References

```{bibliography}
:filter: docname in docnames
```


```{index} Exploit, Payload, Shellcode, Buffer overflow, ASLR, DEP/NX, Stack canary, ROP, Privilege escalation, Lateral movement, Pass-the-hash, Pass-the-ticket, Persistence, Meterpreter
```
